# **Cosmological inference from the BAO-fit results for data**

This notebook shows how to run cosmological inference under $\Lambda$CDM from the BAO-fit results on data (vaying $hr_{\rm d}$ and $\Omega_{\rm m}$).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["PATH"] = "/global/common/software/nersc9/texlive/2024/bin/x86_64-linux:" + os.environ["PATH"]
os.environ["OMP_NUM_THREADS"] = "1"
import sys
from paths import code_path, save_path
sys.path.append(code_path)
import numpy as np
import pandas as pd
from utils_data import GetThetaLimits
from utils_baofit import BAOFitInitializer
from utils_inference import BAOFitChecker, BAOInference
%matplotlib inline

# Option A. DESY6-related stuff
dataset_list = ["DESY6"]
# dataset_list = ["DESY6_dec_above-23.5"]
dynamical_theta_limits = False
old_template = "_old"
delta_z = None # only for DESI stuff
recon = "" # only for DESI stuff
pow_broadband = [-2, -1, 0]

# # Option B. DESI-related stuff
# delta_z = 0.02
# recon = "_recon"
# suffix_list = ["_LRG1", "_LRG2", "_LRG3"]
# dataset_list = [
#     f"DESIY1_LRG_Abacus_altmtl{recon}_deltaz{delta_z}{suffix}"
#     for suffix in suffix_list
# ]
# dynamical_theta_limits = True
# pow_broadband = [-1, 0]

for dataset in dataset_list:

    # Default
    dataset_orig = dataset
    bins_removed = []

    if delta_z is not None:
        if delta_z == 0.028:
            if "DESIY1" in dataset:
                if "LRG1" in dataset:
                    bins_removed = list(map(int, np.arange(7, 25)))  # LRG1
                    dataset_orig = dataset.replace("_LRG1", "")
                elif "LRG2" in dataset:
                    bins_removed = list(map(int, np.concatenate([np.arange(0, 7), np.arange(14, 25)])))  # LRG2
                    dataset_orig = dataset.replace("_LRG2", "")
                elif "LRG3" in dataset:
                    bins_removed = list(map(int, np.arange(0, 14)))  # LRG3
                    dataset_orig = dataset.replace("_LRG3", "")
        elif delta_z == 0.02:
            if "DESIY1" in dataset:
                if "LRG1" in dataset:
                    bins_removed = list(map(int, np.arange(10, 35)))  # LRG1
                    dataset_orig = dataset.replace("_LRG1", "")
                elif "LRG2" in dataset:
                    bins_removed = list(map(int, np.concatenate([np.arange(0, 10), np.arange(20, 35)])))  # LRG2
                    dataset_orig = dataset.replace("_LRG2", "")
                elif "LRG3" in dataset:
                    bins_removed = list(map(int, np.arange(0, 20)))  # LRG3
                    dataset_orig = dataset.replace("_LRG3", "")

    print(f"Loading the results for the {dataset} data set...")

    if "DESIY1" in dataset:
        nz_flag = "mocks"
        cov_type = "mocks"
        cosmology_template = "desifid"
        cosmology_covariance = "desifid"
        
    elif "DESY6" in dataset:
        nz_flag = "fid"
        cov_type = "cosmolike"
        cosmology_template = "planck" + old_template
        cosmology_covariance = "planck"
        
    theta_min, theta_max = GetThetaLimits(dataset=dataset_orig, nz_flag=nz_flag, dynamical_theta_limits=dynamical_theta_limits, code_path=code_path).get_theta_limits()

    # 1. Arguments
    base_config = {
        "dataset": dataset_orig,
        "weight_type": 1,
        "mock_id": "mean", # it will not be used...
        "nz_flag": nz_flag,
        "cov_type": cov_type,
        "cosmology_template": cosmology_template,
        "cosmology_covariance": cosmology_covariance,
        "delta_theta": 0.2,
        "theta_min": theta_min,
        "theta_max": theta_max,
        "pow_broadband": pow_broadband,
        "bins_removed": bins_removed,
        "alpha_min": 0.8,
        "alpha_max": 1.2,
        # "alpha_type": "alpha_wigg_only",
        "alpha_type": "alpha_wigg_nowigg",
        "code_path": code_path,
        "save_path": save_path,
    }
    
    baofit_checker = BAOFitChecker(**base_config)

    # 2. Run cosmological inference
    inference = BAOInference(
        baofit_checker=baofit_checker,
        nwalkers=32,
        nsteps=5000,
        burnin=1000,
        # overwrite=True,
        verbose=True,
    )

    # MCMC starting point
    initial_guess = [100.0, 0.3]  # h*r_d, Omega_m
    samples, log_probs = inference.run_mcmc(initial_guess)

    # Minimizer for best-fit
    best_fit = inference.run_minimizer(samples, log_probs)

    # Summarize MCMC
    results = inference.summarize_chain(samples)

    # Optionally make triangle plot
    inference.make_triangle_plot(samples, best_fit)
            